# Colab Session: scope-jpeg-paper
Generated from colab-cli history log.

**Session Created**: 2026-11-19 13:07:15
- Endpoint: `gpu-a100-s-ass1c0-29zwmyl6i0kf7`
- Hardware: `A100`

In [ ]:
#!/usr/bin/env python3
"""Run unmodified upstream miccunifi/SCOPE JPEG verification on full COCO 2017 val (5,000 images)."""
from __future__ import annotations

import hashlib
import os
import shutil
import subprocess
import sys
import tarfile
import time
import zipfile
from pathlib import Path

os.environ["PYTHONUNBUFFERED"] = "1"

SCOPE_WEIGHTS_ID = "1M7lHWbNjrckMNBC623Cn5PvBDuDUBlda"
EXPECTED_SHA256 = "01bbb0c35f6846293be046a154b31e3037aa2bdbc4dab98362fcf6d9184d81e2"


def run(cmd: list[str], cwd: Path | None = None) -> None:
    print("+ " + " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def main() -> None:
    t0 = time.perf_counter()
    run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
    usage = shutil.disk_usage("/content")
    free_gb = usage.free / (1024.0**3)
    print(f"[Disk] /content total={usage.total / (1024**3):.1f} GB | free={free_gb:.1f} GB", flush=True)
    if free_gb < 35.0:
        raise RuntimeError(f"INSUFFICIENT_DISK: /content has only {free_gb:.1f} GB free (< 35 GB needed)")

    run([
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "pyyaml", "ultralytics", "pycocotools", "gdown", "scipy", "tqdm", "pillow", "opencv-python-headless",
    ])

    import ultralytics
    import yaml

    print(f"[Env] ultralytics={ultralytics.__version__}", flush=True)

    scope_dir = Path("/content/SCOPE")
    if scope_dir.exists():
        shutil.rmtree(scope_dir)
    run(["git", "clone", "https://github.com/miccunifi/SCOPE.git", str(scope_dir)])
    rev = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(scope_dir), text=True).strip()
    print(f"[Upstream SCOPE] Cloned HEAD={rev}", flush=True)

    ckpt_path = (
        scope_dir
        / "runs"
        / "preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_sevbasis_scratch_FULL_20260701_140557"
        / "best_val.pt"
    )
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    run([sys.executable, "-m", "gdown", "--fuzzy", f"https://drive.google.com/uc?id={SCOPE_WEIGHTS_ID}", "-O", str(ckpt_path)])
    actual_sha = sha256_file(ckpt_path)
    print(f"[Checkpoint] {ckpt_path.name} sha256={actual_sha}", flush=True)
    if actual_sha != EXPECTED_SHA256:
        raise RuntimeError(f"Checkpoint SHA mismatch: {actual_sha} != {EXPECTED_SHA256}")

    coco_dir = Path("/content/coco2017")
    coco_dir.mkdir(parents=True, exist_ok=True)
    val_dir = coco_dir / "val2017"
    ann_file = coco_dir / "annotations" / "instances_val2017.json"

    if not (val_dir.is_dir() and len(list(val_dir.glob("*.jpg"))) >= 5000):
        z_img = coco_dir / "val2017.zip"
        run(["curl", "-L", "--retry", "3", "-sS", "-o", str(z_img), "http://images.cocodataset.org/zips/val2017.zip"])
        with zipfile.ZipFile(z_img, "r") as zf:
            zf.extractall(coco_dir)
        z_img.unlink(missing_ok=True)

    if not ann_file.is_file():
        z_ann = coco_dir / "annotations_trainval2017.zip"
        run([
            "curl", "-L", "--retry", "3", "-sS", "-o", str(z_ann),
            "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
        ])
        with zipfile.ZipFile(z_ann, "r") as zf:
            zf.extractall(coco_dir)
        z_ann.unlink(missing_ok=True)

    print(f"[COCO] val2017 images={len(list(val_dir.glob('*.jpg')))}, ann={ann_file.exists()}", flush=True)

    ds_val_dir = scope_dir / "datasets" / "coco-2017" / "validation"
    ds_val_dir.mkdir(parents=True, exist_ok=True)
    data_link = ds_val_dir / "data"
    labels_link = ds_val_dir / "labels.json"
    if data_link.exists() or data_link.is_symlink():
        data_link.unlink()
    if labels_link.exists() or labels_link.is_symlink():
        labels_link.unlink()
    data_link.symlink_to(val_dir)
    labels_link.symlink_to(ann_file)

    models_dir = scope_dir / "models"
    models_dir.mkdir(parents=True, exist_ok=True)
    prev_cwd = Path.cwd()
    try:
        os.chdir(models_dir)
        from ultralytics import RTDETR, YOLO

        YOLO("yolo26x.pt")
        RTDETR("rtdetr-l.pt")
    except Exception as exc:
        raise RuntimeError(f"MISSING_YOLO26X_OR_RTDETR: {exc}") from exc
    finally:
        os.chdir(prev_cwd)

    if not (models_dir / "yolo26x.pt").is_file() or not (models_dir / "rtdetr-l.pt").is_file():
        raise RuntimeError(f"MISSING_YOLO26X: model weights missing in {models_dir}")
    print("[Detectors] Ready: models/yolo26x.pt and models/rtdetr-l.pt", flush=True)

    cfg = yaml.safe_load((scope_dir / "configs" / "eval.yaml").read_text(encoding="utf-8"))
    jpeg_exps = [e for e in cfg["experiments"] if str(e.get("name", "")).startswith(("jpeg_", "scope_jpeg_"))]
    if len(jpeg_exps) != 12:
        raise RuntimeError(f"Expected 12 JPEG experiments, got {len(jpeg_exps)}")
    cfg["experiments"] = jpeg_exps
    eval_jpeg_yaml = scope_dir / "configs" / "eval_jpeg_only.yaml"
    eval_jpeg_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    print(f"[Config] Wrote {eval_jpeg_yaml} with {len(jpeg_exps)} JPEG experiments", flush=True)

    # 1. Upstream YOLO26x + JPEG generation
    run([sys.executable, "-u", "evaluation/evaluate_yolo26x_codecs.py", "--config", "configs/eval_jpeg_only.yaml"], cwd=scope_dir)

    # 2. Upstream RT-DETR-l on saved reconstructions
    run([
        sys.executable, "-u", "evaluation/evaluate_rtdetr_l_saved.py",
        "--source-eval-dir", "outputs_scope_full_main",
        "--images-dir", "datasets/coco-2017/validation/data",
        "--annotations-json", "datasets/coco-2017/validation/labels.json",
        "--detector-weights", "models/rtdetr-l.pt",
        "--category-map", "coco91",
        "--output-dir", "outputs_scope_full_rtdetr_l",
    ], cwd=scope_dir)

    # 3. Upstream Faster R-CNN R50-FPN v2 on saved reconstructions
    run([
        sys.executable, "-u", "-m", "scope.evaluation.saved_multi_detector",
        "--source-eval-dir", "outputs_scope_full_main",
        "--images-dir", "datasets/coco-2017/validation/data",
        "--annotations-json", "datasets/coco-2017/validation/labels.json",
        "--backend", "torchvision",
        "--detector", "fasterrcnn_resnet50_fpn_v2",
        "--label-space", "coco91",
        "--output-dir", "outputs_scope_full_fasterrcnn_v2",
    ], cwd=scope_dir)

    # 4. Upstream BDBR calculation (AP@[.50:.95] default + AP@50)
    run([
        sys.executable, "-u", "analysis/compute_bdbr.py",
        "--detector-dir",
        "full:yolo=outputs_scope_full_main",
        "full:rtdetr=outputs_scope_full_rtdetr_l",
        "full:fastrcnn=outputs_scope_full_fasterrcnn_v2",
        "--annotations-json", "datasets/coco-2017/validation/labels.json",
        "--table-methods", "full",
        "--out-dir", "outputs_scope_bdbr",
    ], cwd=scope_dir)

    run([
        sys.executable, "-u", "analysis/compute_bdbr.py",
        "--detector-dir",
        "full:yolo=outputs_scope_full_main",
        "full:rtdetr=outputs_scope_full_rtdetr_l",
        "full:fastrcnn=outputs_scope_full_fasterrcnn_v2",
        "--annotations-json", "datasets/coco-2017/validation/labels.json",
        "--metric", "AP50",
        "--table-methods", "full",
        "--out-dir", "outputs_scope_bdbr_ap50",
    ], cwd=scope_dir)

    # Archive compact results (CSV, JSON summaries, BDBR outputs, omitting huge PNGs & raw predictions)
    out_tar = Path("/content/upstream_scope_jpeg_results.tar.gz")
    with tarfile.open(out_tar, "w:gz") as tf:
        for dname in ("outputs_scope_bdbr", "outputs_scope_bdbr_ap50"):
            dpath = scope_dir / dname
            if dpath.exists():
                tf.add(str(dpath), arcname=dname)
        for out_root in ("outputs_scope_full_main", "outputs_scope_full_rtdetr_l", "outputs_scope_full_fasterrcnn_v2"):
            rpath = scope_dir / out_root
            if not rpath.exists():
                continue
            for p in sorted(rpath.rglob("*")):
                if p.is_file() and p.name in ("all_results.csv", "summary.json"):
                    tf.add(str(p), arcname=p.relative_to(scope_dir).as_posix())

    print(f"[DONE] Archived {out_tar} ({out_tar.stat().st_size / 1024:.1f} KB) in {time.perf_counter() - t0:.1f}s", flush=True)


if __name__ == "__main__":
    main()


+ nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


[Disk] /content total=235.7 GB | free=235.6 GB


+ /usr/bin/python3 -m pip install -q -U pyyaml ultralytics pycocotools gdown scipy tqdm pillow opencv-python-headless


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


[Env] ultralytics=8.4.215


+ git clone https://github.com/miccunifi/SCOPE.git /content/SCOPE


[Upstream SCOPE] Cloned HEAD=453a40bebd73d4a7c3ad4436370e96f412a0a546


+ /usr/bin/python3 -m gdown --fuzzy https://drive.google.com/uc?id=1M7lHWbNjrckMNBC623Cn5PvBDuDUBlda -O /content/SCOPE/runs/preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_sevbasis_scratch_FULL_20260701_140557/best_val.pt


CalledProcessError: Command '['/usr/bin/python3', '-m', 'gdown', '--fuzzy', 'https://drive.google.com/uc?id=1M7lHWbNjrckMNBC623Cn5PvBDuDUBlda', '-O', '/content/SCOPE/runs/preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_sevbasis_scratch_FULL_20260701_140557/best_val.pt']' returned non-zero exit status 2.

*File Operation*: `ls` on `content/SCOPE`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_main`

*File Operation*: `ls` on `content/SCOPE`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_main`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_rtdetr_l`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/all_results.csv`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_rtdetr_l`

*File Operation*: `download` on `content/SCOPE/configs/eval_jpeg_only.yaml`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/scope_jpeg_q30/bandwidth.csv`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_rtdetr_l`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_rtdetr_l`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_rtdetr_l`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_fasterrcnn_v2`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_rtdetr_l/all_results.csv`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_fasterrcnn_v2`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_fasterrcnn_v2`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_fasterrcnn_v2`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_fasterrcnn_v2`

*File Operation*: `ls` on `content/SCOPE`

*File Operation*: `ls` on `content/SCOPE/outputs_scope_full_fasterrcnn_v2`

In [ ]:
#!/usr/bin/env python3
"""Run unmodified upstream miccunifi/SCOPE JPEG verification on full COCO 2017 val (5,000 images)."""
from __future__ import annotations

import hashlib
import os
import shutil
import subprocess
import sys
import tarfile
import time
import zipfile
from pathlib import Path

os.environ["PYTHONUNBUFFERED"] = "1"

SCOPE_WEIGHTS_ID = "1M7lHWbNjrckMNBC623Cn5PvBDuDUBlda"
EXPECTED_SHA256 = "01bbb0c35f6846293be046a154b31e3037aa2bdbc4dab98362fcf6d9184d81e2"


def run(cmd: list[str], cwd: Path | None = None) -> None:
    print("+ " + " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def main() -> None:
    t0 = time.perf_counter()
    run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
    usage = shutil.disk_usage("/content")
    free_gb = usage.free / (1024.0**3)
    print(f"[Disk] /content total={usage.total / (1024**3):.1f} GB | free={free_gb:.1f} GB", flush=True)
    if free_gb < 35.0:
        raise RuntimeError(f"INSUFFICIENT_DISK: /content has only {free_gb:.1f} GB free (< 35 GB needed)")

    scope_dir = Path("/content/SCOPE")
    if not (scope_dir / "scope" / "inference.py").is_file():
        run([
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "pyyaml", "ultralytics", "pycocotools", "gdown", "scipy", "tqdm", "pillow", "opencv-python-headless",
        ])
        if scope_dir.exists():
            shutil.rmtree(scope_dir)
        run(["git", "clone", "https://github.com/miccunifi/SCOPE.git", str(scope_dir)])
    else:
        print("[Reuse] Reusing existing /content/SCOPE checkout and installed packages.", flush=True)

    import ultralytics
    import yaml

    print(f"[Env] ultralytics={ultralytics.__version__}", flush=True)
    rev = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(scope_dir), text=True).strip()
    print(f"[Upstream SCOPE] HEAD={rev}", flush=True)

    ckpt_path = (
        scope_dir
        / "runs"
        / "preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_sevbasis_scratch_FULL_20260701_140557"
        / "best_val.pt"
    )
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    if not (ckpt_path.is_file() and sha256_file(ckpt_path) == EXPECTED_SHA256):
        run([
            "curl", "-L", "--retry", "3", "-sS",
            f"https://drive.usercontent.google.com/download?id={SCOPE_WEIGHTS_ID}&export=download&confirm=t",
            "-o", str(ckpt_path),
        ])
    actual_sha = sha256_file(ckpt_path)
    print(f"[Checkpoint] {ckpt_path.name} sha256={actual_sha}", flush=True)
    if actual_sha != EXPECTED_SHA256:
        raise RuntimeError(f"Checkpoint SHA mismatch: {actual_sha} != {EXPECTED_SHA256}")

    coco_dir = Path("/content/coco2017")
    coco_dir.mkdir(parents=True, exist_ok=True)
    val_dir = coco_dir / "val2017"
    ann_file = coco_dir / "annotations" / "instances_val2017.json"

    if not (val_dir.is_dir() and len(list(val_dir.glob("*.jpg"))) >= 5000):
        z_img = coco_dir / "val2017.zip"
        run(["curl", "-L", "--retry", "3", "-sS", "-o", str(z_img), "http://images.cocodataset.org/zips/val2017.zip"])
        with zipfile.ZipFile(z_img, "r") as zf:
            zf.extractall(coco_dir)
        z_img.unlink(missing_ok=True)

    if not ann_file.is_file():
        z_ann = coco_dir / "annotations_trainval2017.zip"
        run([
            "curl", "-L", "--retry", "3", "-sS", "-o", str(z_ann),
            "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
        ])
        with zipfile.ZipFile(z_ann, "r") as zf:
            zf.extractall(coco_dir)
        z_ann.unlink(missing_ok=True)

    print(f"[COCO] val2017 images={len(list(val_dir.glob('*.jpg')))}, ann={ann_file.exists()}", flush=True)

    ds_val_dir = scope_dir / "datasets" / "coco-2017" / "validation"
    ds_val_dir.mkdir(parents=True, exist_ok=True)
    data_link = ds_val_dir / "data"
    labels_link = ds_val_dir / "labels.json"
    if data_link.exists() or data_link.is_symlink():
        data_link.unlink()
    if labels_link.exists() or labels_link.is_symlink():
        labels_link.unlink()
    data_link.symlink_to(val_dir)
    labels_link.symlink_to(ann_file)

    models_dir = scope_dir / "models"
    models_dir.mkdir(parents=True, exist_ok=True)
    prev_cwd = Path.cwd()
    try:
        os.chdir(models_dir)
        from ultralytics import RTDETR, YOLO

        YOLO("yolo26x.pt")
        RTDETR("rtdetr-l.pt")
    except Exception as exc:
        raise RuntimeError(f"MISSING_YOLO26X_OR_RTDETR: {exc}") from exc
    finally:
        os.chdir(prev_cwd)

    if not (models_dir / "yolo26x.pt").is_file() or not (models_dir / "rtdetr-l.pt").is_file():
        raise RuntimeError(f"MISSING_YOLO26X: model weights missing in {models_dir}")
    print("[Detectors] Ready: models/yolo26x.pt and models/rtdetr-l.pt", flush=True)

    cfg = yaml.safe_load((scope_dir / "configs" / "eval.yaml").read_text(encoding="utf-8"))
    jpeg_exps = [e for e in cfg["experiments"] if str(e.get("name", "")).startswith(("jpeg_", "scope_jpeg_"))]
    if len(jpeg_exps) != 12:
        raise RuntimeError(f"Expected 12 JPEG experiments, got {len(jpeg_exps)}")
    cfg["experiments"] = jpeg_exps
    eval_jpeg_yaml = scope_dir / "configs" / "eval_jpeg_only.yaml"
    eval_jpeg_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    print(f"[Config] Wrote {eval_jpeg_yaml} with {len(jpeg_exps)} JPEG experiments", flush=True)

    # 1. Upstream YOLO26x + JPEG generation
    run([sys.executable, "-u", "evaluation/evaluate_yolo26x_codecs.py", "--config", "configs/eval_jpeg_only.yaml"], cwd=scope_dir)

    # 2. Upstream RT-DETR-l on saved reconstructions
    run([
        sys.executable, "-u", "evaluation/evaluate_rtdetr_l_saved.py",
        "--source-eval-dir", "outputs_scope_full_main",
        "--images-dir", "datasets/coco-2017/validation/data",
        "--annotations-json", "datasets/coco-2017/validation/labels.json",
        "--detector-weights", "models/rtdetr-l.pt",
        "--category-map", "coco91",
        "--output-dir", "outputs_scope_full_rtdetr_l",
    ], cwd=scope_dir)

    # 3. Upstream Faster R-CNN R50-FPN v2 on saved reconstructions
    run([
        sys.executable, "-u", "-m", "scope.evaluation.saved_multi_detector",
        "--source-eval-dir", "outputs_scope_full_main",
        "--images-dir", "datasets/coco-2017/validation/data",
        "--annotations-json", "datasets/coco-2017/validation/labels.json",
        "--backend", "torchvision",
        "--detector", "fasterrcnn_resnet50_fpn_v2",
        "--label-space", "coco91",
        "--output-dir", "outputs_scope_full_fasterrcnn_v2",
    ], cwd=scope_dir)

    # 4. Upstream BDBR calculation (AP@[.50:.95] default + AP@50)
    run([
        sys.executable, "-u", "analysis/compute_bdbr.py",
        "--detector-dir",
        "full:yolo=outputs_scope_full_main",
        "full:rtdetr=outputs_scope_full_rtdetr_l",
        "full:fastrcnn=outputs_scope_full_fasterrcnn_v2",
        "--annotations-json", "datasets/coco-2017/validation/labels.json",
        "--table-methods", "full",
        "--out-dir", "outputs_scope_bdbr",
    ], cwd=scope_dir)

    run([
        sys.executable, "-u", "analysis/compute_bdbr.py",
        "--detector-dir",
        "full:yolo=outputs_scope_full_main",
        "full:rtdetr=outputs_scope_full_rtdetr_l",
        "full:fastrcnn=outputs_scope_full_fasterrcnn_v2",
        "--annotations-json", "datasets/coco-2017/validation/labels.json",
        "--metric", "AP50",
        "--table-methods", "full",
        "--out-dir", "outputs_scope_bdbr_ap50",
    ], cwd=scope_dir)

    for bdbr_csv in (
        scope_dir / "outputs_scope_bdbr" / "bdbr_summary.csv",
        scope_dir / "outputs_scope_bdbr_ap50" / "bdbr_summary.csv",
    ):
        if bdbr_csv.is_file():
            print(f"\n=== {bdbr_csv.relative_to(scope_dir)} ===\n" + bdbr_csv.read_text(encoding="utf-8"), flush=True)

    # Archive compact results (CSV, JSON summaries, BDBR outputs, omitting huge PNGs & raw predictions)
    out_tar = Path("/content/upstream_scope_jpeg_results.tar.gz")
    with tarfile.open(out_tar, "w:gz") as tf:
        for dname in ("outputs_scope_bdbr", "outputs_scope_bdbr_ap50"):
            dpath = scope_dir / dname
            if dpath.exists():
                tf.add(str(dpath), arcname=dname)
        for out_root in ("outputs_scope_full_main", "outputs_scope_full_rtdetr_l", "outputs_scope_full_fasterrcnn_v2"):
            rpath = scope_dir / out_root
            if not rpath.exists():
                continue
            for p in sorted(rpath.rglob("*")):
                if p.is_file() and p.name in ("all_results.csv", "summary.json"):
                    tf.add(str(p), arcname=p.relative_to(scope_dir).as_posix())

    print(f"[DONE] Archived {out_tar} ({out_tar.stat().st_size / 1024:.1f} KB) in {time.perf_counter() - t0:.1f}s", flush=True)


if __name__ == "__main__":
    main()


+ nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


[Disk] /content total=235.7 GB | free=235.1 GB


[Reuse] Reusing existing /content/SCOPE checkout and installed packages.


[Env] ultralytics=8.4.215


[Upstream SCOPE] HEAD=453a40bebd73d4a7c3ad4436370e96f412a0a546


+ curl -L --retry 3 -sS https://drive.usercontent.google.com/download?id=1M7lHWbNjrckMNBC623Cn5PvBDuDUBlda&export=download&confirm=t -o /content/SCOPE/runs/preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_sevbasis_scratch_FULL_20260701_140557/best_val.pt


... (10 outputs left out)

+ /usr/bin/python3 -u analysis/compute_bdbr.py --detector-dir full:yolo=outputs_scope_full_main full:rtdetr=outputs_scope_full_rtdetr_l full:fastrcnn=outputs_scope_full_fasterrcnn_v2 --annotations-json datasets/coco-2017/validation/labels.json --metric AP50 --table-methods full --out-dir outputs_scope_bdbr_ap50



=== outputs_scope_bdbr/bdbr_summary.csv ===
codec,detector,test_method,reference_method,bdbr_percent,common_accuracy_min,common_accuracy_max,reference_pareto_points,test_pareto_points,status
jpeg,Faster R-CNN,SCOPE,Baseline,-16.485293187857984,0.11567238735025767,0.3867640269074642,6,6,ok
jpeg,RT-DETR,SCOPE,Baseline,-5.622940934220555,0.3677956880724293,0.4681377273878892,6,6,ok
jpeg,YOLO,SCOPE,Baseline,-14.981235476054145,0.2556399767534386,0.4943719140326179,6,6,ok
bpg,Faster R-CNN,SCOPE,Baseline,,,,,,missing_reference_curve
bpg,RT-DETR,SCOPE,Baseline,,,,,,missing_reference_curve
bpg,YOLO,SCOPE,Baseline,,,,,,missing_reference_curve
bmshj,Faster R-CNN,SCOPE,Baseline,,,,,,missing_reference_curve
bmshj,RT-DETR,SCOPE,Baseline,,,,,,missing_reference_curve
bmshj,YOLO,SCOPE,Baseline,,,,,,missing_reference_curve




=== outputs_scope_bdbr_ap50/bdbr_summary.csv ===
codec,detector,test_method,reference_method,bdbr_percent,common_accuracy_min,common_accuracy_max,reference_pareto_points,test_pareto_points,status
jpeg,Faster R-CNN,SCOPE,Baseline,-16.418625437650945,0.18693277558708984,0.5626083692393316,6,6,ok
jpeg,RT-DETR,SCOPE,Baseline,-3.5508547531731938,0.5328512870876271,0.6456625154936083,6,6,ok
jpeg,YOLO,SCOPE,Baseline,-14.57886119401197,0.3768629289536488,0.6614821775514171,6,6,ok
bpg,Faster R-CNN,SCOPE,Baseline,,,,,,missing_reference_curve
bpg,RT-DETR,SCOPE,Baseline,,,,,,missing_reference_curve
bpg,YOLO,SCOPE,Baseline,,,,,,missing_reference_curve
bmshj,Faster R-CNN,SCOPE,Baseline,,,,,,missing_reference_curve
bmshj,RT-DETR,SCOPE,Baseline,,,,,,missing_reference_curve
bmshj,YOLO,SCOPE,Baseline,,,,,,missing_reference_curve



[DONE] Archived /content/upstream_scope_jpeg_results.tar.gz (25.9 KB) in 12034.5s


*File Operation*: `download` on `content/upstream_scope_jpeg_results.tar.gz`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/jpeg_q10/bandwidth.csv`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/scope_jpeg_q10/bandwidth.csv`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/jpeg_q15/bandwidth.csv`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/scope_jpeg_q15/bandwidth.csv`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/jpeg_q20/bandwidth.csv`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/scope_jpeg_q20/bandwidth.csv`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/jpeg_q30/bandwidth.csv`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/scope_jpeg_q30/bandwidth.csv`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/jpeg_q40/bandwidth.csv`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/scope_jpeg_q40/bandwidth.csv`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/jpeg_q50/bandwidth.csv`

*File Operation*: `download` on `content/SCOPE/outputs_scope_full_main/scope_jpeg_q50/bandwidth.csv`